In [37]:
using Pkg                # Juliaのパッケージ管理機能（Pkgモジュール）を読み込む
Pkg.activate("..")       # カレントディレクトリの「1つ上のフォルダ」にあるProject.toml をアクティブにする（notebooks/ からリポジトリ直下の環境を使うため）
using Revise             # src/ 以下のファイルを編集したときに，Juliaを再起動しなくても変更を自動で反映してくれる開発用ツール
using Horizons3DViewer2  # 今作っている自作パッケージ Horizons3DViewer2 を読み込む（src/Horizons3DViewer2.jl がロードされる）
using EphemerisSources
using AstroTime, Dates
using GLMakie

  Activating project at `~/work/Horizons3DViewer2`


https://juliaastro.org/EphemerisSources.jl/docs/stable/examples/telescope-study/
https://juliaastro.org/AstroTime/stable/tutorial/#Tutorial
https://ssd.jpl.nasa.gov/horizons/app.html#/

In [38]:
######################################
# 取得したいデータを指定
######################################

####################
frame = "J2000"
wrt   = "sun"
units = "KM-S"

####################
#spacecraft = "Orion EM-1"
#start = TDBEpoch(2022, 11, 16, 9, 4, 0.0)
#stop  = TDBEpoch(2022, 12, 11, 17, 21, 0.0)
#step  = "30 min"

####################
#spacecraft = "CAPSTONE"
#start = TDBEpoch(2022, 6, 28, 10, 6, 7.0)
#stop  = start + 150days
#step  = "1 hour"

####################
#spacecraft = "Hayabusa 2"
#start = TDBEpoch(2014, 12, 3, 6, 14, 54.0)
#stop  = TDBEpoch(2026, 4, 10, 0, 0, 0.0)
#step  = "1 day"
#ryugu_horizons = ephemeris("162173", start, stop, step ; wrt = wrt, units = units, frame = freme)
#ryugu_states = horizons_to_matrix(ryugu_horizons)

####################
spacecraft = "Europa Clipper" #全体
start = TDBEpoch(2024, 10, 14, 16, 15, 4.0)
stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
step  = "1 day"
#step  = "2 hour"
europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
europa_states = horizons_to_matrix(europa_horizons)

####################
#spacecraft = "Europa Clipper" # 木星に入ってから
#start = TDBEpoch(2030, 4, 1, 0, 0, 0.0)
#start = TDBEpoch(2031, 1, 1, 0, 0, 0.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#stop  = TDBEpoch(2032, 1, 1, 0, 0, 0.0)
#step  = "1 hour"
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)

######################################
# データの取得して扱いやすい形に変換
######################################

# TDBで指定する。
earth_horizons   = ephemeris("earth", start, stop, step ; wrt = wrt, units = units, frame = frame)
mars_horizons    = ephemeris("mars", start, stop, step ; wrt = wrt, units = units, frame = frame)
jupiter_horizons = ephemeris("jupiter", start, stop, step ; wrt = wrt, units = units, frame = frame)
moon_horizons    = ephemeris("moon", start, stop, step ; wrt = wrt, units = units, frame = frame)
sc1_horizons     = ephemeris(spacecraft, start, stop, step ; wrt = wrt, units = units, frame = frame)

earth_states     = horizons_to_matrix(earth_horizons)
mars_states      = horizons_to_matrix(mars_horizons)
jupiter_states   = horizons_to_matrix(jupiter_horizons)
moon_states      = horizons_to_matrix(moon_horizons)
sc1_states       = horizons_to_matrix(sc1_horizons)

t_tdb       = TDBEpoch.(moon_horizons.t .* days; origin = :julian)
t_tai       = TAIEpoch.(t_tdb)

println(size(moon_states))
println(size(sc1_states))
println(moon_states[1:6,1])
println(sc1_states[1:6,1])
println(t_tdb[1])
println(t_tai[1])

(6, 3611)
(6, 3611)
[1.391809358769481e8, 5.44889020893749e7, -17434.01018048823, -11.09890941220298, 28.63786251385828, 0.08997216082347492]
[1.388309909173316e8, 5.45996046899446e7, -418.0671930275857, -9.778723071271738, 19.96624456629233, 0.2955908787900698]
2024-10-14T16:15:04.000 TDB
2024-10-14T16:14:31.817 TAI


In [39]:
######################################
# 座標変換
######################################

# EME2000への座標変換
moon_states_eme = eclJ2000_to_EME2000(moon_states-earth_states)
sc1_states_eme  = eclJ2000_to_EME2000(sc1_states-earth_states)

# 地球月固定回転座標系(地球中心)への変換
moon_em_fixed_e = make_line_fixed_frame(moon_states-earth_states, moon_states-earth_states)
sc1_em_fixed_e  = make_line_fixed_frame(moon_states-earth_states, sc1_states-earth_states)

# 地球月固定回転座標系(月中心)への変換
sc1_em_fixed_m  = sc1_em_fixed_e - moon_em_fixed_e

# 太陽地球固定回転座標系への変換
earth_se_fixed_s = make_line_fixed_frame(earth_states, earth_states)
moon_se_fixed_e  = make_line_fixed_frame(earth_states, moon_states) - earth_se_fixed_s
sc1_se_fixed_e   = make_line_fixed_frame(earth_states, sc1_states) - earth_se_fixed_s

# 白道面への変換
moon_lop_e      = rotate_to_plane_fixed_t0(moon_states-earth_states, moon_states-earth_states) #lunar orbital plane
sc1_lop_e       = rotate_to_plane_fixed_t0(moon_states-earth_states, sc1_states-earth_states)


3×3611 Matrix{Float64}:
 -5599.84       -2.97889e5  -5.99758e5  …   5.95949e8   5.95162e8   5.94726e8
  -757.938  478970.0         9.43035e5      1.27362e8   1.25632e8   1.23915e8
  3305.68    60335.6         1.25986e5     -2.2559e7   -2.24302e7  -2.22942e7

In [40]:
######################################
# 必要時に未来の軌道を伝播
######################################

show_propagated_flag = 0

if show_propagated_flag == 1
    N = size(earth_states,2)
    step_size_sec = 24*60*60

    results_dict = Dict{Int, NamedTuple{(:sol, :sol_em_fixed_e, :sol_lop_e), Tuple{Any, Any, Any}}}()

    for sol_from in 1:sampling_interval:(N-1)
        sol = solve_satellite(sc1_states, earth_states, moon_states, mars_states, jupiter_states;
            sol_from=sol_from, step_size_sec=step_size_sec, reltol=1e-10, abstol=1e-10)

        sol_N = size(sol,2)

        rel_moon_earth = moon_states[1:6, sol_from:sol_from+sol_N-1] .- earth_states[1:6, sol_from:sol_from+sol_N-1]
        rel_sc_earth   = sol[1:6, :]                  .- earth_states[1:6, sol_from:sol_from+sol_N-1]

        results_dict[sol_from] = (
            sol            = sol,
            sol_em_fixed_e = make_line_fixed_frame(rel_moon_earth, rel_sc_earth),
            sol_lop_e      = rotate_to_plane_fixed_t0(rel_moon_earth, rel_sc_earth)
        )
    end
end


In [44]:
function default_axis3(parent;
    title::AbstractString,
    viewmode = :fit,
    clip = false,
    titlegap = -50,
    hide_decorations::Bool = true,
    azimuth = -0.5pi,
    elevation = 0.5pi,
    scale = nothing,              # ← 追加
    offset = (0.0, 0.0, 0.0)       # ← 追加（xだけずらす用途対応）
    )

    ax = Axis3(parent;
        aspect = :data,
        xlabel = "x", ylabel = "y",
        viewmode = viewmode,
        clip = clip,
        title = title,
        titlegap = titlegap,
        xspinesvisible = false,
        yspinesvisible = false,
        zspinesvisible = false,
        xticksvisible = false,
        yticksvisible = false,
        zticksvisible = false
    )

    # -------------------------
    # decorations on/off
    # -------------------------
    if hide_decorations
        hidedecorations!(ax)
    end

    # -------------------------
    # view (後からも上書き可能)
    # -------------------------
    if azimuth !== nothing
        ax.azimuth[] = azimuth
    end
    if elevation !== nothing
        ax.elevation[] = elevation
    end

    # -------------------------
    # scale / limits
    # -------------------------
    if scale !== nothing
        ox, oy, oz = offset
        limits!(ax,
            (-scale + ox, scale + ox),
            (-scale + oy, scale + oy),
            (-scale + oz, scale + oz)
        )
    end

    return ax
end

default_axis3 (generic function with 1 method)

In [45]:
using ProgressMeter

#描画領域の設定
set_theme!(theme_black()) # ダークテーマを設定
update_theme!(fontsize= 22)

fig1 = Figure(size = (1400, 800))
fig2 = Figure(size = (1400, 800))

# fig1
ax_ecliptic_s = default_axis3(fig1[1, 1];
    scale = 6e8,
    title = "Fig.X Sun centered, Ecliptic of J2000",
)

ax_ecliptic_j = default_axis3(fig1[1, 2];
    scale = 3e6,
    title = "Fig.X Jupiter centered, Ecliptic of J2000",
)

# fig2（※衝突しないように配置を整理）
ax_lop_e = default_axis3(fig2[1, 1];
    scale = 9e5,  # 元コードは 3e5 と 9e5 が両方あったのでどちらかに確定
    title = "Fig.1 Inertial frame",
)

ax_em_fixed_e = default_axis3(fig2[1, 2];
    scale  = 2e5,
    offset = (2e5, 0.0, 0.0),   # 元コードの x 方向 +2e5 を再現
    title  = "Fig.2 Earth-Moon line fixed rotational frame",
)

ax_ecliptic_e = default_axis3(fig2[1, 3];
    scale = 3e4,
    title = "Fig.X Earth centered, Ecliptic of J2000",
)

ax_se_fixed_e = default_axis3(fig2[2, 1];
    title = "Fig.X Earth centered, Sun-Earth line fixed rotational frame",
)

# 元コードでは ax_ecliptic_europa が fig2[1,2] で衝突していたので fig2[2,2] に移動
ax_ecliptic_europa = default_axis3(fig2[2, 2];
    scale = 2e4,
    title = "Fig.X Europa centered, Ecliptic of J2000",
)

# 元コードでは ax_eme_e が fig2[1,3] で衝突していたので fig2[2,3] に移動
ax_eme_e = default_axis3(fig2[2, 3];
    title = "Fig.X Earth centered, EME2000",
)

# 元コードでは ax_em_fixed_m が fig2[2,3] でさらに衝突していたので、3行目へ
ax_em_fixed_m = default_axis3(fig2[3, 1];
    scale = 7e4,
    title = "Fig.X Moon centered, Earth-Moon line fixed rotational frame",
)

#=
title_ecliptic_s      = "Fig.X Sun centered, Ecliptic of J2000"
title_ecliptic_e      = "Fig.X Earth centered, Ecliptic of J2000"
title_ecliptic_j      = "Fig.X Jupiter centered, Ecliptic of J2000"
title_ecliptic_europa = "Fig.X Europa centered, Ecliptic of J2000"
title_eme_e           = "Fig.X Earth centered, EME2000"
title_lop_e           = "Fig.1 Inertial frame"
title_se_fixed_e      = "Fig.X Earth centered, Sun-Earth line fixed rotational frame"
title_em_fixed_e      = "Fig.2 Earth-Moon line fixed rotational frame"
title_em_fixed_m      = "Fig.x Moon centered, Earth-Moon line fixed rotational frame"

ax_lop_e            = Axis3(fig2[1, 1], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_lop_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_em_fixed_e       = Axis3(fig2[1, 2], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_em_fixed_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_em_fixed_m       = Axis3(fig2[2, 3], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_em_fixed_m, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_s       = Axis3(fig1[1, 1], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_s, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_e       = Axis3(fig2[1, 3], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_j       = Axis3(fig1[1, 2], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_j, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_europa  = Axis3(fig2[1, 2], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_europa, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_eme_e            = Axis3(fig2[1, 3], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_eme_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_se_fixed_e       = Axis3(fig2[2, 1], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_se_fixed_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)

hidedecorations!(ax_lop_e)
hidedecorations!(ax_em_fixed_e)
hidedecorations!(ax_ecliptic_s)
hidedecorations!(ax_ecliptic_j)
hidedecorations!(ax_ecliptic_europa)

ax_lop_e.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_lop_e.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π

ax_em_fixed_e.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_em_fixed_e.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π

ax_ecliptic_s.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_s.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π
ax_ecliptic_e.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_e.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π
ax_ecliptic_j.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_j.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π
ax_ecliptic_europa.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_europa.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π

scale_ecliptic_s = 6e8
scale_ecliptic_e = 3e4
scale_ecliptic_j = 3e6
scale_ecliptic_europa = 2e4
scale_lop_e = 3e5
scale_lop_e = 9e5
scale_em_fixed_e = 2e5
scale_em_fixed_m = 7e4

limits!(ax_ecliptic_s, (-scale_ecliptic_s, scale_ecliptic_s), (-scale_ecliptic_s, scale_ecliptic_s), (-scale_ecliptic_s, scale_ecliptic_s))
limits!(ax_ecliptic_e, (-scale_ecliptic_e, scale_ecliptic_e), (-scale_ecliptic_e, scale_ecliptic_e), (-scale_ecliptic_e, scale_ecliptic_e))
limits!(ax_ecliptic_j, (-scale_ecliptic_j, scale_ecliptic_j), (-scale_ecliptic_j, scale_ecliptic_j), (-scale_ecliptic_j, scale_ecliptic_j))
limits!(ax_ecliptic_europa, (-scale_ecliptic_europa, scale_ecliptic_europa), (-scale_ecliptic_europa, scale_ecliptic_europa), (-scale_ecliptic_europa, scale_ecliptic_europa))
limits!(ax_lop_e, (-scale_lop_e, scale_lop_e), (-scale_lop_e, scale_lop_e), (-scale_lop_e, scale_lop_e))
limits!(ax_em_fixed_e, (-scale_em_fixed_e+2e5, scale_em_fixed_e+2e5), (-scale_em_fixed_e, scale_em_fixed_e), (-scale_em_fixed_e, scale_em_fixed_e))
limits!(ax_em_fixed_m, (-scale_em_fixed_m, scale_em_fixed_m), (-scale_em_fixed_m, scale_em_fixed_m), (-scale_em_fixed_m, scale_em_fixed_m))
=#

######### 描画 ##################################################

sun_sphere   = Sphere(Point3f(0.0, 0.0, 0.0), 696340)
earth_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 6378)
moon_sphere  = Sphere(Point3f(0.0, 0.0, 0.0), 1737)
jupiter_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 69911)
europa_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 1600)

#mesh!(ax_ecliptic_s, sun_sphere, color=:white)
mesh!(ax_ecliptic_e, earth_sphere, color=:cyan)
mesh!(ax_eme_e, earth_sphere, color=:cyan)
mesh!(ax_lop_e, earth_sphere, color=:cyan)
mesh!(ax_se_fixed_e, earth_sphere, color=:cyan)
mesh!(ax_em_fixed_e, earth_sphere, color=:cyan)
mesh!(ax_em_fixed_m, moon_sphere, color=:yellow)
mesh!(ax_ecliptic_j, jupiter_sphere, color=:orange)
mesh!(ax_ecliptic_europa, europa_sphere, color=:cyan)
scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0, markersize=10, color=:white)

animation_flag = 1
sampling_interval1  = 60
#sampling_interval2 = 60
N = size(sc1_states, 2)
frames = 1:sampling_interval1:(N-1)

# 切り替えフレーム（ここから細かくする）
#switch_frame = Int(floor(0.6 * N))   # 例：後半20%を細かく
#frames1 = 1:sampling_interval1:(switch_frame-1)
#frames2 = switch_frame:sampling_interval2:(N-1)
#frames = vcat(frames1, frames2)

if animation_flag == 0

    lines!(ax_ecliptic_s, earth_states[1:3, :], linewidth=1, color=:cyan)
    lines!(ax_ecliptic_s, mars_states[1:3, :], linewidth=1, color=:red)
    lines!(ax_ecliptic_s, jupiter_states[1:3, :], linewidth=1, color=:orange)
    lines!(ax_ecliptic_s, sc1_states[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_ecliptic_e, moon_states[1:3, :]- earth_states[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :],  linewidth=1, color=:lime)
    scatter!(ax_ecliptic_e, sc1_states[1:3, :]'- earth_states[1:3, :]',  markersize=10, color=:lime)
    
    lines!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:lime)
    lines!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:white)

    lines!(ax_eme_e, moon_states_eme[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_eme_e, sc1_states_eme[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_lop_e, moon_lop_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_lop_e, sc1_lop_e[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_lop_e, results_dict[651].sol_lop_e[1:3, :],  linewidth=1, color=:red)

    lines!(ax_se_fixed_e, moon_se_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_se_fixed_e, sc1_se_fixed_e[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_em_fixed_e, moon_em_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_em_fixed_e, sc1_em_fixed_e[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_em_fixed_e, results_dict[651].sol_em_fixed_e[1:3, :],  linewidth=1, color=:red)

    lines!(ax_em_fixed_m, sc1_em_fixed_m[1:3, :],  linewidth=1, color=:lime)

else
    frame_0   = Observable(0)
    frame_0[] = 1
    trail_len_long = 12000
    trail_len_short = 1200

    if show_propagated_flag == 1
        lines!(ax_ecliptic_s, @lift(results_dict[$frame_0].sol[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_lop_e, @lift(results_dict[$frame_0].sol_lop_e[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_em_fixed_e, @lift(results_dict[$frame_0].sol_em_fixed_e[1:3, :]),  linewidth=1, color=:red)
    end

    text!(ax_ecliptic_s, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 22)

    traj_fade!(ax_ecliptic_s, earth_states[1:3, :], frame_0; trail_len=trail_len_long, color=:cyan)
    traj_fade!(ax_ecliptic_s, mars_states[1:3, :], frame_0; trail_len=trail_len_long, color=:pink)
    traj_fade!(ax_ecliptic_s, jupiter_states[1:3, :], frame_0; trail_len=trail_len_long, color=:orange)
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len_long, color=:lime)
    #traj_fade!(ax_ecliptic_s, ryugu_states[1:3, :], frame_0; trail_len=trail_len_long, color=:orange)

    traj_fade!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:white)

    traj_fade!(ax_lop_e, moon_lop_e, frame_0; trail_len=trail_len_short, color=:yellow)    
    traj_fade!(ax_lop_e, sc1_lop_e, frame_0; trail_len=trail_len_short, color=:lime)

    #moon_sphere_ine = lift((x, y, z) -> Sphere(Point3f(x, y, z), moon_radius*5),
    #@lift(Float32(moon_int[1, $frame_0])),
    #@lift(Float32(moon_int[2, $frame_0])),
    #@lift(Float32(moon_int[3, $frame_0])))
    #mesh!(ax_ine, moon_sphere_ine, color=:yellow)

    p = Progress(length(frames); desc = "Rendering movie")

    # アニメ出力 (interpolationed) 
    record(fig1, "../output/$(spacecraft)_5.mp4", frames;framerate = 30) do frame
    #framerateは1秒間のフレーム数(60fpsが標準的な上限) #地球火星の軌道を描くための設定(frame_planetを更新) 

        frame_0[] = frame
        next!(p)   # ← これを1行追加

    end

end

fig1


Rendering movie 100%|████████████████████████████████████| Time: 0:00:02
